# Distributed reaction–diffusion on four MPI ranks
**Student ID 230103054** · MPI + CUDA · 512 × 512 interior cells

GitHub: `ibragimovmardandev-tech`

Select a T4 runtime and run all cells. The last cell exports the notebook with the measured results, the solver, and the manifest.

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
%pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-3b2514c0-4ae8-1064-2f2b-a47f9a103706, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



In [2]:
import subprocess

GITHUB_USERNAME = "ibragimovmardandev-tech"

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"],
    text=True,
).strip()
if "T4" not in gpu_name:
    raise RuntimeError(f"Select a Tesla T4 runtime in Colab. Detected GPU: {gpu_name}")

print(f"Student ID: 230103054 | GitHub: {GITHUB_USERNAME} | GPU: {gpu_name}")

Student ID: 230103054 | GitHub: ibragimovmardandev-tech | GPU: Tesla T4


In [3]:
%%writefile distributed_cuda_solver.py
import hashlib
import json
import re
import subprocess
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

TILE_DIM = 16
SHARED_DIM = 18
GLOBAL_NX = GLOBAL_NY = 512
N = GLOBAL_NX // 2
DX = 1 / (GLOBAL_NX + 1)
STEPS = 400
LAMBDA_DAMP = 0.15
OPPOSITE = (1, 0, 3, 2)


def get_gpu_uuid():
    return subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0].strip()


def derive_student_seed(github_user, gpu_uuid):
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode()).hexdigest()
    return int(digest[:16], 16)


def init_distributed_context(user):
    world = MPI.COMM_WORLD
    if world.size != 4:
        raise ValueError("Launch exactly four MPI ranks.")
    cuda.select_device(0)
    cart = world.Create_cart([2, 2], periods=[False, False], reorder=False)
    uuid = cart.bcast(get_gpu_uuid() if cart.rank == 0 else None, root=0)
    return cart, (*cart.Shift(0, 1), *cart.Shift(1, 1)), uuid, derive_student_seed(user, uuid)


def initialize_subdomains(coords, neighbors, seed):
    row, col = coords
    y = (row * N + np.arange(N + 2)[:, None]) * DX
    x = (col * N + np.arange(N + 2)[None, :]) * DX
    rng = np.random.default_rng(seed)
    k = 0.16 + 0.08 * np.cos(2*np.pi*x) * np.sin(2*np.pi*y)
    u = np.zeros_like(k)
    for _ in range(3):
        cx, cy = rng.uniform(0.15, 0.85, 2)
        u += rng.uniform(0.8, 1.4) * np.exp(-((x-cx)**2 + (y-cy)**2)/0.035)
    s = 0.0001 * np.exp(-((x-0.5)**2 + (y-0.5)**2)/0.01)
    for side, neighbor in enumerate(neighbors):
        if neighbor == MPI.PROC_NULL:
            if side < 2:
                u[(0, -1)[side], :] = 0
            else:
                u[:, (0, -1)[side-2]] = 0
    return u, k, s


@cuda.jit
def pack_halos_kernel(u, buf):
    t = cuda.grid(1)
    if t < N:
        buf[t] = u[1, t+1]
        buf[N+t] = u[N, t+1]
        buf[2*N+t] = u[t+1, 1]
        buf[3*N+t] = u[t+1, N]


@cuda.jit
def unpack_halos_kernel(u, buf):
    t = cuda.grid(1)
    if t < N:
        u[0, t+1] = buf[t]
        u[N+1, t+1] = buf[N+t]
        u[t+1, 0] = buf[2*N+t]
        u[t+1, N+1] = buf[3*N+t]


@cuda.jit
def tiled_stencil_kernel(u, out, k, source, dt, r_start, r_end, c_start, c_end):
    tile = cuda.shared.array((SHARED_DIM, SHARED_DIM), float64)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    sx, sy = tx+1, ty+1
    j = c_start + cuda.blockIdx.x*TILE_DIM + tx
    i = r_start + cuda.blockIdx.y*TILE_DIM + ty
    # Padding lanes load the positive apron for a partial block. They join
    # the barrier but never read an MPI ghost while the deep interior runs.
    r, c = min(i, r_end+1), min(j, c_end+1)
    tile[sy, sx] = u[r, c]
    if tx == 0:
        tile[sy, 0] = u[r, c-1]
    if ty == 0:
        tile[0, sx] = u[r-1, c]
    if tx == 15:
        tile[sy, sx+1] = u[r, min(c+1, c_end+1)]
    if ty == 15:
        tile[sy+1, sx] = u[min(r+1, r_end+1), c]
    cuda.syncthreads()
    if i <= r_end and j <= c_end:
        v = tile[sy, sx]
        kc = k[i, j]
        flux = ((kc+k[i, j+1])*(tile[sy, sx+1]-v)
                + (kc+k[i, j-1])*(tile[sy, sx-1]-v)
                + (kc+k[i+1, j])*(tile[sy+1, sx]-v)
                + (kc+k[i-1, j])*(tile[sy-1, sx]-v))
        out[i, j] = v + dt*(0.5*flux/(DX*DX) - LAMBDA_DAMP*v*v*v + source[i, j])


def exchange_halos_async(cart, neighbors, u, boundary, send, host_send, host_recv):
    pack_halos_kernel[2, 128, boundary](u, send)
    send.copy_to_host(host_send, stream=boundary)
    boundary.synchronize()
    requests = []
    for side, neighbor in enumerate(neighbors):
        section = slice(side*N, (side+1)*N)
        requests.append(cart.Irecv(host_recv[section], neighbor, OPPOSITE[side]))
        requests.append(cart.Isend(host_send[section], neighbor, side))
    return requests


def launch_box(args, stream, r0, r1, c0, c1):
    grid = ((c1-c0+16)//16, (r1-r0+16)//16)
    tiled_stencil_kernel[grid, (16, 16), stream](*args, r0, r1, c0, c1)


def step_overlapped(cart, neighbors, args, streams, buffers, overlap):
    interior, boundary = streams
    send, recv, host_send, host_recv = buffers
    requests = exchange_halos_async(cart, neighbors, args[0], boundary, send, host_send, host_recv)
    if overlap:
        launch_box(args, interior, 2, N-1, 2, N-1)
    MPI.Request.Waitall(requests)
    recv.copy_to_device(host_recv, stream=boundary)
    unpack_halos_kernel[2, 128, boundary](args[0], recv)
    if not overlap:
        launch_box(args, boundary, 2, N-1, 2, N-1)
    for box in ((1, 1, 1, N), (N, N, 1, N), (2, N-1, 1, 1), (2, N-1, N, N)):
        launch_box(args, boundary, *box)
    interior.synchronize()
    boundary.synchronize()


def metrics(cart, current, previous, step):
    a = current.copy_to_host()[1:-1, 1:-1]
    b = previous.copy_to_host()[1:-1, 1:-1]
    local = np.array([a.sum()*DX**2, ((a-b)**2).sum()*DX**2])
    total = np.empty_like(local)
    cart.Allreduce(local, total, op=MPI.SUM)
    return dict(step=step, energy=float(total[0]), l2_residual=float(np.sqrt(total[1])))


def run_solver(user):
    cart, neighbors, uuid, seed = init_distributed_context(user)
    coords = cart.Get_coords(cart.rank)
    u0, kh, sh = initialize_subdomains(coords, neighbors, seed)
    kmax = cart.allreduce(kh[1:-1, 1:-1].max(), op=MPI.MAX)
    dt = 0.8 * DX**2 / (4*kmax)
    k, source = cuda.to_device(kh), cuda.to_device(sh)
    streams = cuda.stream(), cuda.stream()
    buffers = (cuda.device_array(4*N, np.float64), cuda.device_array(4*N, np.float64),
               cuda.pinned_array(4*N, np.float64), cuda.pinned_array(4*N, np.float64))
    buffers[-1].fill(0)  # PROC_NULL receives leave the destination unchanged.
    print(f"rank={cart.rank} coords={coords} neighbors={neighbors} shape={u0.shape}", flush=True)
    record = dict(github_username=user, student_id="230103054", gpu_uuid=uuid,
                  instance_seed=seed, mpi_ranks=cart.size, cart_dims=[2, 2],
                  local_shape=list(u0.shape), halo_bytes=N*u0.itemsize, steps=STEPS, dx=DX, dt=dt, kappa_max=kmax)
    # Warm all kernels and both execution paths before measuring.
    a, b = cuda.to_device(u0), cuda.to_device(u0)
    for mode in (False, True):
        step_overlapped(cart, neighbors, (a, b, k, source, dt), streams, buffers, mode)
    ptx = next(iter(tiled_stencil_kernel.inspect_asm().values()))
    counts = {op: len(re.findall(r"\b"+re.escape(op)+r"\b", ptx))
              for op in ("st.shared.f64", "ld.shared.f64", "bar.sync", "st.shared", "ld.shared")}
    record.update(ptx_counts=counts, ptx_audit_passed=all(counts[t] > 0 for t in ("st.shared", "ld.shared", "bar.sync")))
    for mode, label in ((False, "serialized"), (True, "overlapped")):
        a.copy_to_device(u0)
        b.copy_to_device(u0)
        trace = [metrics(cart, a, b, 0)]
        wall_ms = gpu_ms = 0.
        start_event, stop_event = cuda.event(), cuda.event()
        for start in range(0, STEPS, 100):
            cart.Barrier()
            start_event.record(streams[1])
            tick = perf_counter()
            for _ in range(100):
                step_overlapped(cart, neighbors, (a, b, k, source, dt), streams, buffers, mode)
                a, b = b, a
            wall_ms += (perf_counter()-tick)*1000
            stop_event.record(streams[1])
            stop_event.synchronize()
            gpu_ms += cuda.event_elapsed_time(start_event, stop_event)
            trace.append(metrics(cart, a, b, start+100))
        record[label+"_ms"] = cart.allreduce(wall_ms, op=MPI.MAX)
        record[label+"_cuda_event_ms"] = cart.allreduce(gpu_ms, op=MPI.MAX)
        record[label+"_trace"] = trace
        final = a.copy_to_host()[1:-1, 1:-1]
        if not mode:
            baseline = final.copy()
    diff = cart.allreduce(float(np.max(np.abs(final-baseline))), op=MPI.MAX)
    record["parity_max_diff"] = diff
    if not record["ptx_audit_passed"] or not diff < 1e-12:
        raise RuntimeError(f"Verification failed: {counts}, parity={diff}")
    if not all(np.isfinite(item["energy"]) and np.isfinite(item["l2_residual"])
               for item in trace):
        raise RuntimeError("Non-finite diagnostics")
    if cart.rank == 0:
        output = json.dumps(record, indent=2, allow_nan=False)
        print(output, flush=True)
        Path("submission_manifest.json").write_text(output+"\n")


if __name__ == "__main__":
    run_solver(sys.argv[1])

Writing distributed_cuda_solver.py


In [4]:
import subprocess
from pathlib import Path
Path("submission_manifest.json").unlink(missing_ok=True)
cmd = ["mpirun", "--allow-run-as-root", "--oversubscribe", "-np", "4",
       "python3", "distributed_cuda_solver.py", GITHUB_USERNAME]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.stderr:
    print("STDERR:", proc.stderr)
proc.check_returncode()

rank=2 coords=[1, 0] neighbors=(0, -2, -2, 3) shape=(258, 258)
rank=0 coords=[0, 0] neighbors=(-2, 2, -2, 1) shape=(258, 258)
rank=3 coords=[1, 1] neighbors=(1, -2, 2, -2) shape=(258, 258)
rank=1 coords=[0, 1] neighbors=(-2, 3, 0, -2) shape=(258, 258)
{
  "github_username": "ibragimovmardandev-tech",
  "student_id": "230103054",
  "gpu_uuid": "GPU-3b2514c0-4ae8-1064-2f2b-a47f9a103706",
  "instance_seed": 6707915264826495882,
  "mpi_ranks": 4,
  "cart_dims": [
    2,
    2
  ],
  "local_shape": [
    258,
    258
  ],
  "halo_bytes": 2048,
  "steps": 400,
  "dx": 0.001949317738791423,
  "dt": 3.1665577794689985e-06,
  "kappa_max": 0.2399981248663085,
  "ptx_counts": {
    "st.shared.f64": 5,
    "ld.shared.f64": 5,
    "bar.sync": 1,
    "st.shared": 5,
    "ld.shared": 5
  },
  "ptx_audit_passed": true,
  "serialized_ms": 1352.2779889999867,
  "serialized_cuda_event_ms": 1352.7412719726562,
  "serialized_trace": [
    {
      "step": 0,
      "energy": 0.3406666147374898,
      "l2_res

### Q1 · Why pack the columns?

The local field has shape 258 × 258 and stores 8-byte doubles in row-major order. Two successive entries in a column are 258 × 8 = 2,064 bytes apart. When adjacent warp lanes read that column, they access separate memory sectors. The packing kernel writes those values side by side in the send buffer, so DMA can transfer the column in one 2048-byte copy.

The first and last entries are 255 strides apart. A contiguous copy covering both would span (255 × 258 + 1) × 8 = 526,328 bytes. Subtracting the 2,048 bytes actually needed gives **524,280 excess bytes per column**, or **1,048,560 bytes** for two separate column copies per rank.

### Q2 · Shared tile and bank mapping
The 258 × 258 padded field is tiled with 16 × 16 threads. In a full block, thread (0,0) writes center [1,1], west [1,0] and north [0,1]. Its five reads are [1,1], [1,0], [1,2], [0,1], [2,1]. Thread (15,15) writes center [16,16], east [16,17] and south [17,16]. It reads [16,16], [16,15], [16,17], [15,16], [17,16]. Apron corners are unused.

A double at [r,c] occupies banks b and b+1, with b = 2(18r+c) mod 32. A warp spans two rows of 16 lanes. Each group of 16 adjacent doubles uses every bank once, so the two 128-byte portions of a row-wise warp load are intrinsic to its width. They do not imply a two-way conflict. In a hypothetical column-wise access, successive doubles advance 36 bank words, or 4 modulo 32. The bank pair repeats every eight lanes, giving a two-way conflict within a 16-lane group. Reading the north or south neighbor in this kernel still addresses adjacent columns across lanes. [NVIDIA bank-transaction discussion](https://forums.developer.nvidia.com/t/how-to-understand-the-bank-conflict-of-shared-mem/260900)

### Q3 · What does the block barrier protect?

The compiled kernel contains 5 `st.shared.f64`, 5 `ld.shared.f64` and 1 `bar.sync` instructions. Across all PTX types there are 5 shared stores and 5 shared loads. The audit returned `True`. These are counts in the compiled listing. A running block can execute an instruction many times.

Consider the thread at (tx=0, ty=1). It reads its southern neighbor from tile[3,1]. The thread that writes that value is (0,2), which belongs to the next warp because each warp covers two rows of 16 threads. Without a barrier the reader could reach tile[3,1] before the writer fills it. The barrier covers all eight warps, including the threads that only load padding in a partial block.

### Q4 · Four processes on one T4
For 400 steps, the slowest rank took 1352.278 ms serialized and 1245.272 ms overlapped (ratio 1.0859×, time saving 7.913%). CUDA-event elapsed times were 1352.741 and 1245.050 ms, respectively. The CUDA events include time spent waiting between kernels. Overlap reduced wall time in this run. These timings measure the combined cost of computation, copies and communication.

The four ranks use separate CUDA contexts on the same T4. Under conventional scheduling without MPS, the GPU time-slices contexts. Streams within a context can overlap eligible kernels and DMA, while all ranks share SMs, memory bandwidth and copy engines. Host MPI progress may overlap deep-interior work, but context switches, short messages and launch overhead can offset the gain. Four dedicated GPUs would permit simultaneous rank kernels on independent hardware. Across InfiniBand the network latency, bandwidth and possible device-direct transfers would also change the useful overlap window. [NVIDIA CUDA MPS overview](https://docs.nvidia.com/deploy/pdf/CUDA_Multi_Process_Service_Overview.pdf)

### Q5 · Deadlock and physical boundaries
Each active neighbor message has 2048 bytes. Standard blocking sends can complete eagerly if MPI has buffer space. Above the transport-dependent eager threshold, MPI may use a rendezvous that waits for a matching receive. For example, if every rank attempts *all* sends in up/down/left/right order before posting receives, (0,0) skips its null up neighbor and sends down to (1,0), whose first real send goes up to (0,0). Both can wait for receives neither has reached. This threshold is implementation-dependent and was not measured. Posting nonblocking receives and sends before Waitall removes that circular dependency.

At (0,0), up and left are MPI.PROC_NULL. Sends and receives targeting them complete without transferring data, and receives leave the pinned buffer untouched. Initializing those slots to zero and unpacking them at every step enforces homogeneous Dirichlet conditions on both exterior edges.

### Q6 · Stability and measured thermal energy
The measured maximum owned-cell diffusivity is 0.239998124866. With Δx = 1/513, Δtmax = Δx²/(4κmax) = 3.95819722434e-06; Δt = 3.16655777947e-06, or 0.800 of that bound. Each face uses the mean diffusivity of its two adjacent cells. Even the largest possible value in this field (0.24) gives a limit of about 3.9582 × 10⁻⁶. For positive u, cubic damping removes heat and the source adds heat. Summing the conservative fluxes cancels internal faces in pairs and leaves boundary flux, damping and source. Hence a diffusion CFL bound alone does not prove monotonically decreasing total heat.

| Step | Thermal energy | L2 change |
|---:|---:|---:|
| 0 | 0.34066661473749 | 0 |
| 100 | 0.339043121916702 | 6.260027232e-05 |
| 200 | 0.338203076687168 | 3.877306307e-05 |
| 300 | 0.337519944965876 | 3.003965357e-05 |
| 400 | 0.336917695368399 | 2.550609548e-05 |

Energy was non-increasing at every recorded checkpoint. The L2 column measures the area-weighted norm of the change since the immediately preceding timestep. It is zero by convention at initialization. The two final fields differ by at most 0. The trace tests this particular seeded run at the recorded checkpoints.